# TP 1 — Premiers réseaux de neurones avec `keras`

Ce TP accompagne les chapitres 1 à 3 du [poly du cours](https://rtavenar.github.io/deep_sns) : lisez-les avant de commencer.

**Objectifs :**
1. programmer vous-même quelques pas de **descente de gradient** ;
2. découvrir la bibliothèque `keras` et ses trois étapes : **définir**, **compiler**, **entraîner** un modèle ;
3. comparer un neurone seul et des **perceptrons multicouches** (MLP) sur un vrai problème : la reconnaissance de chiffres manuscrits.

**Mode d'emploi :**
* exécutez les cellules **dans l'ordre**, avec `Maj + Entrée` ;
* les zones à compléter sont signalées par `# À COMPLÉTER` et contiennent `...` : remplacez les `...` par votre code ;
* certaines cellules commencent par `# VÉRIFICATION` : elles vous disent si votre réponse semble correcte ;
* si quelque chose ne marche plus, `Exécution > Redémarrer la session et tout exécuter` (dans Colab) repart de zéro.

In [ ]:
# Imports : exécutez cette cellule en premier
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras.models import Sequential
from keras.layers import Input, Dense

print("Version de keras :", keras.__version__)

## Partie 1 — La descente de gradient, à la main

On reprend l'exemple du poly : prédire le prix des logements d'un quartier de Boston (en milliers de dollars) à partir du nombre moyen de pièces par logement, avec le modèle à un seul paramètre

$$\text{prix prédit} = w \times \text{nombre de pièces}$$

La cellule suivante charge les données et les affiche.

In [ ]:
from keras.datasets import boston_housing

(x_boston, y_boston), _ = boston_housing.load_data()
pieces = x_boston[:, 5]  # la 6e colonne contient le nombre moyen de pièces
prix = y_boston

print("Nombre de quartiers :", len(prix))
plt.scatter(pieces, prix, alpha=.5)
plt.xlabel("Nombre moyen de pièces")
plt.ylabel("Prix médian (milliers de $)");

Les deux fonctions ci-dessous sont fournies :
* `perte(w)` calcule l'erreur quadratique moyenne du modèle pour une valeur de `w` ;
* `pente(w)` calcule la pente de la courbe de perte au point `w` (c'est le **gradient**).

In [ ]:
def perte(w):
    return np.mean((w * pieces - prix) ** 2)

def pente(w):
    return np.mean(2 * (w * pieces - prix) * pieces)

print("Perte pour w = 2 :", perte(2.))
print("Pente pour w = 2 :", pente(2.))

**Question 1.** Complétez la fonction `descente_de_gradient` ci-dessous.
Partant de `w_init`, elle doit effectuer `n_etapes` pas de descente de gradient avec le taux d'apprentissage `taux`, et renvoyer la liste de toutes les valeurs successives de `w`.

_Rappel : à chaque étape, on se déplace dans le sens opposé à la pente, d'un pas proportionnel au taux d'apprentissage._

In [ ]:
def descente_de_gradient(w_init, taux, n_etapes):
    w = w_init
    valeurs = [w]
    for etape in range(n_etapes):
        w = w - taux * pente(w)
        valeurs.append(w)
    return valeurs

In [ ]:
# VÉRIFICATION
valeurs = descente_de_gradient(w_init=0., taux=0.005, n_etapes=20)
if isinstance(valeurs[-1], float) and abs(valeurs[-1] - 3.64) < 0.05:
    print(f"✅ Bravo ! Après 20 étapes, w vaut {valeurs[-1]:.2f}.")
else:
    print("❌ Pas encore : après 20 étapes avec un taux de 0.005, on devrait obtenir w ≈ 3.64.")

La fonction `trace_descente` (fournie) affiche la courbe de perte et les étapes de la descente.

**Question 2.** Utilisez-la pour afficher 20 étapes de descente de gradient, à partir de `w_init = 0`, pour les taux d'apprentissage `0.0005`, `0.005` et `0.0255`.
Que constatez-vous dans chacun des cas ?

In [ ]:
def trace_descente(valeurs, titre=""):
    ww = np.linspace(-4, 11, 200)
    plt.plot(ww, [perte(w) for w in ww], color="orange", alpha=.6)
    plt.plot(valeurs, [perte(w) for w in valeurs], "ko-")
    plt.xlabel("w")
    plt.ylabel("Perte")
    plt.title(titre)
    plt.show()

In [ ]:
for taux in [0.0005, 0.005, 0.0255]:
    valeurs = descente_de_gradient(w_init=0., taux=taux, n_etapes=20)
    trace_descente(valeurs, titre=f"Taux d'apprentissage : {taux}")

**Réponse.** * Avec `0.0005`, le taux est **trop petit** : on avance dans la bonne direction, mais très lentement (après 20 étapes, on est encore loin du minimum, autour de w ≈ 2).
* Avec `0.005`, le taux est **bien choisi** : on atteint le minimum (w ≈ 3,64) en une dizaine d'étapes.
* Avec `0.0255`, le taux est **trop grand** : on saute d'un côté à l'autre de la vallée en s'éloignant un peu plus à chaque fois, l'algorithme **diverge**.

## Partie 2 — Découvrir `keras`

Dans `keras`, utiliser un réseau de neurones se fait toujours en trois étapes.

**1. Définir le modèle**, en empilant des couches (voir le chapitre 3 du poly) :
```python
model = Sequential([
    Input(shape=(10, )),                  # 10 variables en entrée
    Dense(units=20, activation="relu"),   # une couche cachée de 20 neurones
    Dense(units=3, activation="softmax")  # 3 neurones de sortie
])
```

**2. Compiler le modèle**, c'est-à-dire choisir :
* la **fonction de perte** à minimiser (`loss`) : `"mse"` (erreur quadratique moyenne) pour une régression, `"categorical_crossentropy"` pour une classification à plusieurs classes (nous verrons en séance 2 d'où vient cette perte) ;
* l'**optimiseur** (`optimizer`), c'est-à-dire la méthode utilisée pour minimiser la perte : `"sgd"` est la descente de gradient de la partie 1, `"adam"` est une variante plus perfectionnée, qui fonctionne bien dans la plupart des cas ;
* éventuellement des **métriques** à afficher pendant l'entraînement, comme le taux de bonnes réponses (`"accuracy"`).
```python
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
```

**3. Entraîner le modèle** sur des données :
```python
model.fit(x_train, y_train, epochs=10, batch_size=100)
```
* une **époque** (`epoch`) correspond à un passage complet sur toutes les données d'entraînement ;
* `batch_size` est le nombre d'exemples utilisés pour calculer chaque pas de gradient : plutôt que de calculer la pente sur toutes les données à chaque pas (comme dans la partie 1), on la calcule sur un petit paquet d'exemples, ce qui est beaucoup plus rapide.

### Les données

Nous allons utiliser deux jeux de données, préparés par les fonctions ci-dessous (fournies) :
* `prepare_boston()` : les logements de Boston, avec cette fois les **13** variables disponibles ;
* `prepare_mnist()` : des images de chiffres manuscrits (MNIST).

Ces fonctions renvoient les données d'entraînement (`x_train`, `y_train`) et de test (`x_test`, `y_test`), et mettent toutes les valeurs à la même échelle (entre 0 et 1), ce qui facilite l'apprentissage.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from keras.datasets import mnist
from keras.utils import to_categorical


def prepare_boston():
    (x_train, y_train), (x_test, y_test) = boston_housing.load_data()
    # mise à l'échelle des variables explicatives
    scaler_x = MinMaxScaler().fit(x_train)
    x_train, x_test = scaler_x.transform(x_train), scaler_x.transform(x_test)
    # mise à l'échelle de la cible (le prix)
    scaler_y = MinMaxScaler().fit(y_train.reshape((-1, 1)))
    y_train = scaler_y.transform(y_train.reshape((-1, 1)))
    y_test = scaler_y.transform(y_test.reshape((-1, 1)))
    return x_train, x_test, y_train, y_test


def prepare_mnist():
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    # chaque image 28x28 est « aplatie » en une liste de 784 pixels
    x_train = x_train.reshape((x_train.shape[0], -1)) / 255.
    x_test = x_test.reshape((x_test.shape[0], -1)) / 255.
    # chaque chiffre (0 à 9) est codé par 10 valeurs : 1 pour la bonne classe, 0 ailleurs
    y_train = to_categorical(y_train)
    y_test = to_categorical(y_test)
    return x_train, x_test, y_train, y_test


def affiche_chiffres(x, y, n=10):
    plt.figure(figsize=(2 * n, 2.5))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(x[i].reshape((28, 28)), cmap="gray_r")
        plt.title(f"Classe : {np.argmax(y[i])}")
        plt.axis("off")
    plt.show()

**Question 3.** Exécutez la cellule suivante, puis répondez :
1. Combien y a-t-il d'exemples d'entraînement et de variables explicatives dans chaque jeu de données ?
2. Lequel correspond à un problème de régression ? De classification ?
3. Pourquoi `y_train` a-t-il 10 colonnes pour MNIST ?

In [ ]:
x_train, x_test, y_train, y_test = prepare_boston()
print("Boston :", x_train.shape, y_train.shape)

x_train, x_test, y_train, y_test = prepare_mnist()
print("MNIST  :", x_train.shape, y_train.shape)
print("Les 3 premières lignes de y_train :")
print(y_train[:3])

affiche_chiffres(x_train, y_train)

**Réponse.** 1. Boston : 404 exemples d'entraînement décrits par 13 variables. MNIST : 60 000 images d'entraînement, chacune décrite par 784 variables (les 28 × 28 pixels).
2. Boston est un problème de **régression** (on prédit un prix) ; MNIST est un problème de **classification** à 10 classes (les chiffres de 0 à 9).
3. Chaque chiffre est codé par 10 valeurs : 1 dans la colonne de la bonne classe, 0 ailleurs. Cela correspond aux 10 neurones de sortie du modèle, chacun donnant la probabilité d'une classe.

## Partie 3 — Un seul neurone

**Question 4.** Sur les données de Boston, définissez un **perceptron** (aucune couche cachée, un seul neurone de sortie, activation `"linear"`), compilez-le pour minimiser l'erreur quadratique moyenne avec la descente de gradient `"sgd"`, et entraînez-le pendant 10 époques avec des paquets (`batch_size`) de 32 exemples.

Observez la perte (`loss`) affichée à chaque époque : diminue-t-elle ?

In [ ]:
x_train, x_test, y_train, y_test = prepare_boston()

model = Sequential([
    Input(shape=(13, )),
    Dense(units=1, activation="linear")
])
model.compile(loss="mse", optimizer="sgd")
model.fit(x_train, y_train, epochs=10, batch_size=32)

**Réponse.** La perte diminue d'époque en époque : `keras` effectue exactement ce que nous avons programmé dans la partie 1 (des pas de descente de gradient), mais avec 13 poids et un biais au lieu d'un seul paramètre.

**Question 5.** Sur les données MNIST, définissez de même un modèle **sans couche cachée** : la couche de sortie doit avoir un neurone par classe, avec l'activation adaptée à une classification à plusieurs classes (voir le chapitre 3 du poly).
Compilez-le avec la perte `"categorical_crossentropy"`, l'optimiseur `"adam"` et la métrique `"accuracy"`, puis entraînez-le pendant 10 époques avec des paquets de 100 exemples.

Quel taux de bonnes réponses (`accuracy`) obtenez-vous à la fin de l'entraînement ?

In [ ]:
x_train, x_test, y_train, y_test = prepare_mnist()

model = Sequential([
    Input(shape=(784, )),
    Dense(units=10, activation="softmax")
])
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model.fit(x_train, y_train, epochs=10, batch_size=100)

**Réponse.** On obtient environ 93 % de bonnes réponses sur les données d'entraînement : c'est déjà bien, mais un chiffre sur 14 environ est mal classé.

## Partie 4 — Des perceptrons multicouches

**Question 6.** Toujours sur MNIST, comparez le modèle précédent avec des MLP ayant respectivement **1**, **2** et **3 couches cachées** de 128 neurones chacune, avec l'activation `"relu"`.
Utilisez les mêmes réglages que précédemment pour `compile` et `fit`.

Notez le taux de bonnes réponses obtenu à la fin de l'entraînement dans le tableau ci-dessous.

| Modèle | Accuracy (entraînement) |
|---|---|
| Sans couche cachée | |
| 1 couche cachée | |
| 2 couches cachées | |
| 3 couches cachées | |

In [ ]:
# 1 couche cachée
model1 = Sequential([
    Input(shape=(784, )),
    Dense(units=128, activation="relu"),
    Dense(units=10, activation="softmax")
])
model1.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model1.fit(x_train, y_train, epochs=10, batch_size=100)

In [ ]:
# 2 couches cachées
model2 = Sequential([
    Input(shape=(784, )),
    Dense(units=128, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=10, activation="softmax")
])
model2.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model2.fit(x_train, y_train, epochs=10, batch_size=100)

In [ ]:
# 3 couches cachées
model3 = Sequential([
    Input(shape=(784, )),
    Dense(units=128, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=10, activation="softmax")
])
model3.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
model3.fit(x_train, y_train, epochs=10, batch_size=100)

**Réponse.** Les MLP font nettement mieux que le modèle sans couche cachée (environ 99 % de bonnes réponses sur l'entraînement contre 93 %).
Les écarts entre 1, 2 et 3 couches cachées sont faibles ici… mais attention : ces chiffres sont mesurés sur les données d'**entraînement**, ce qui n'est pas une bonne façon de comparer des modèles (voir la partie 5).

**Question 7.** Calculez **à la main** le nombre de paramètres du MLP à une couche cachée (`model1`) : 784 entrées, 128 neurones cachés, 10 neurones de sortie.
Reportez votre résultat dans la cellule ci-dessous, puis vérifiez-le avec `model1.count_params()`.

_Indice : refaites l'exercice #1 du chapitre 3 du poly._

In [ ]:
nb_params_calcule = (784 * 128 + 128) + (128 * 10 + 10)
print(nb_params_calcule)

In [ ]:
# VÉRIFICATION
if nb_params_calcule == model1.count_params():
    print(f"✅ Exact : {model1.count_params()} paramètres.")
else:
    print(f"❌ Pas tout à fait : keras compte {model1.count_params()} paramètres. "
          "Avez-vous pensé aux biais ?")

Pour aller plus loin, `model3.summary()` affiche le détail couche par couche :

In [ ]:
model3.summary()

## Partie 5 — Mieux comparer les modèles : le jeu de validation

Comparer des modèles sur les données qui ont servi à les entraîner est une mauvaise idée : un modèle peut très bien « apprendre par cœur » ses exemples d'entraînement sans être capable de bien prédire sur de nouvelles données.

Pour comparer des modèles, on met donc de côté une partie des données d'entraînement, appelée **jeu de validation**, que le modèle ne voit pas pendant l'apprentissage.
Dans `keras`, il suffit d'ajouter l'argument `validation_split` à `fit` : avec `validation_split=0.3`, les derniers 30 % des données d'entraînement sont mis de côté, et `keras` affiche à chaque époque la perte et l'accuracy sur ces données (`val_loss` et `val_accuracy`).

La méthode `fit` renvoie un **historique** de l'entraînement, que la fonction `trace_historique` (fournie) permet de visualiser.

In [ ]:
def trace_historique(historique, titre=""):
    h = historique.history
    epoques = np.arange(1, len(h["loss"]) + 1)
    plt.plot(epoques, h["accuracy"], "o-", label="Entraînement")
    plt.plot(epoques, h["val_accuracy"], "o-", label="Validation")
    plt.xlabel("Époque")
    plt.ylabel("Accuracy")
    plt.title(titre)
    plt.legend()
    plt.show()

**Question 8.** Entraînez à nouveau le MLP à **1 couche cachée** et celui à **3 couches cachées**, en utilisant cette fois 30 % des données d'entraînement comme jeu de validation.
Stockez le résultat de `fit` dans une variable (par exemple `historique1 = model1.fit(...)`) et visualisez-le avec `trace_historique`.

Quel modèle obtient la meilleure accuracy **en validation** ? Que remarquez-vous en comparant les courbes d'entraînement et de validation ?

In [ ]:
model1 = Sequential([
    Input(shape=(784, )),
    Dense(units=128, activation="relu"),
    Dense(units=10, activation="softmax")
])
model1.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
historique1 = model1.fit(x_train, y_train, epochs=10, batch_size=100, validation_split=0.3)
trace_historique(historique1, "1 couche cachée")

In [ ]:
model3 = Sequential([
    Input(shape=(784, )),
    Dense(units=128, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=128, activation="relu"),
    Dense(units=10, activation="softmax")
])
model3.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])
historique3 = model3.fit(x_train, y_train, epochs=10, batch_size=100, validation_split=0.3)
trace_historique(historique3, "3 couches cachées")

**Réponse.** Les deux modèles atteignent une accuracy de validation proche (autour de 97 %), nettement inférieure à leur accuracy d'entraînement (qui approche 99-100 %).
L'écart entre les deux courbes se creuse au fil des époques : les modèles continuent de progresser sur les données d'entraînement, mais plus (ou presque plus) sur les données de validation. C'est le signe qu'ils commencent à **sur-apprendre** (apprendre par cœur), un phénomène que nous étudierons en séance 3.

## Partie 6 — Retour sur le taux d'apprentissage

Dans `keras`, on peut régler le taux d'apprentissage de l'optimiseur en passant un objet à la place du simple texte `"sgd"` :
```python
from keras.optimizers import SGD

model.compile(loss="categorical_crossentropy", optimizer=SGD(learning_rate=0.01), metrics=["accuracy"])
```

**Question 9.** Entraînez le MLP à une couche cachée avec l'optimiseur `SGD` et les taux d'apprentissage `0.0001`, `0.1` et `10.0`, en gardant 30 % des données pour la validation, et visualisez les historiques.
Retrouvez-vous les trois comportements observés dans la partie 1 ?

_Astuce : pour éviter de recopier trois fois le même code, utilisez une boucle `for taux in [0.0001, 0.1, 10.0]:`._

In [ ]:
from keras.optimizers import SGD

for taux in [0.0001, 0.1, 10.0]:
    model = Sequential([
        Input(shape=(784, )),
        Dense(units=128, activation="relu"),
        Dense(units=10, activation="softmax")
    ])
    model.compile(loss="categorical_crossentropy", optimizer=SGD(learning_rate=taux), metrics=["accuracy"])
    historique = model.fit(x_train, y_train, epochs=10, batch_size=100, validation_split=0.3, verbose=0)
    trace_historique(historique, f"Taux d'apprentissage : {taux}")

**Réponse.** Oui :
* avec `0.0001`, le taux est trop petit : l'accuracy progresse très lentement ;
* avec `0.1`, l'apprentissage se passe bien : on atteint rapidement une bonne accuracy ;
* avec `10.0`, le taux est beaucoup trop grand : l'apprentissage diverge et le modèle ne fait pas mieux que le hasard (accuracy autour de 10 %, voire une perte qui vaut `nan`, c'est-à-dire qui a « explosé »).

## Bonus — Utiliser le modèle

Une fois un modèle choisi (grâce au jeu de validation), on mesure ses performances **une seule fois**, sur le jeu de **test**, avec la méthode `evaluate`.
On peut aussi lui demander des prédictions sur de nouvelles images avec `predict`, qui renvoie pour chaque image les 10 probabilités de la couche de sortie.

**Question 10 (bonus).** Évaluez `model1` sur le jeu de test, puis affichez les prédictions du modèle pour les 10 premières images de test (la classe prédite est celle de plus grande probabilité : `np.argmax`).

In [ ]:
perte_test, accuracy_test = model1.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy sur le jeu de test : {accuracy_test:.1%}")

probas = model1.predict(x_test[:10], verbose=0)
print("Classes prédites :", np.argmax(probas, axis=1))
affiche_chiffres(x_test, y_test)